In [1]:
import pandas as pd
from pathlib import Path
import os
import sys

# força o Spark a usar o MESMO python (.venv) pro worker que roda o driver —
# sem isso, o worker sobe com o python de C:\spark\ (instalação separada,
# sem pyarrow instalado) e mapInPandas quebra com ModuleNotFoundError
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# evita crash silencioso do worker Python no Windows quando numpy/scipy e
# pyarrow carregam runtimes OpenMP conflitantes no mesmo processo — precisa
# ser setado ANTES da SparkSession, pra propagar aos subprocessos worker
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import numpy as np
from scipy.spatial import ConvexHull, QhullError

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [2]:
# traceback "bonito" do IPython quebra (TokenError) ao formatar erros vindos
# de frames com fonte dinâmica (ex: lambdas de F.filter/F.transform do Spark)
# no Python 3.13 — Plain evita isso e mostra o erro real
%xmode Plain

Exception reporting mode: Plain


In [3]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.python.worker.faulthandler.enabled", "true") # se algum worker crashar de novo, imprime o traceback nativo real
    .master("local[*]")
    .appName("feature_engineering")
    .getOrCreate()
)

## Funções

In [4]:
# Cada função de feature recebe o mesmo "context": um dict com um sub-dict
# "groups" (coordenadas já limpas de NaN + centroide, um por grupo de
# jogadores — calculado uma vez por linha em compute_defensive_features, não
# uma vez por feature) e as chaves soltas que não fazem parte de um grupo
# (ball_x/y, stadiumLength). Cada compute_* só lê do que já foi calculado,
# nunca chama _clean_xy/mean() de novo pro mesmo grupo.


def _clean_xy(xs, ys):
    """Remove pares (x, y) com NaN — tracking ausente pra aquele jogador."""
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    valid = ~(np.isnan(xs) | np.isnan(ys))
    return xs[valid], ys[valid]


def _clean_group(xs_raw, ys_raw):
    """
    Limpa e cacheia, uma vez por grupo/linha, as coordenadas (sem NaN) e o
    centroide — evita recalcular _clean_xy/mean() em cada compute_* que usa
    o mesmo grupo (ex: defense_line é usado por compute_defense_width,
    compute_height_goal_def_centroide, compute_def_mid_dist e
    compute_def_atk_dist).
    """
    xs, ys = _clean_xy(xs_raw, ys_raw)
    if len(xs) == 0:
        return {"xs": xs, "ys": ys, "cx": np.nan, "cy": np.nan}
    return {"xs": xs, "ys": ys, "cx": float(xs.mean()), "cy": float(ys.mean())}


# Grupo -> (coluna x, coluna y) achatadas — usado por compute_defensive_features
# pra montar context["groups"] uma vez por linha. Adicionar uma feature que
# precisa de um grupo novo: só adicionar aqui (e no achatamento/TEMP_COLS).
GROUP_DEFS = [
    ("defending_outfield", "defending_outfield_x", "defending_outfield_y"),
    ("defense_line", "defense_line_x", "defense_line_y"),
    ("mid_line", "mid_line_x", "mid_line_y"),
    ("atk_line", "atk_line_x", "atk_line_y"),
    ("defending", "defending_x", "defending_y"),
    ("attacking", "attacking_x", "attacking_y"),
]


# ---------------------------------------------------------------------------
# Forma/dispersão do time defendendo
# ---------------------------------------------------------------------------

def compute_surface_area(context):
    """Feature 1: área do casco convexo dos defensores de linha (m²)."""
    g = context["groups"]["defending_outfield"]
    xs, ys = g["xs"], g["ys"]
    if len(xs) < 3:
        return np.nan
    pts = np.array(sorted(set(zip(xs.tolist(), ys.tolist()))), dtype=float)
    if len(pts) < 3:
        return np.nan
    try:
        area = ConvexHull(pts).volume  # em 2D, .volume = área (.area seria o perímetro)
        return round(float(area), 2)
    except QhullError:
        return np.nan  # pontos colineares -> casco degenerado


def compute_stretch_index(context):
    """Feature 2: distância média dos defensores de linha até o centroide (m)."""
    g = context["groups"]["defending_outfield"]
    xs, ys, cx, cy = g["xs"], g["ys"], g["cx"], g["cy"]
    if len(xs) == 0:
        return np.nan
    stretch_index = np.mean(np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2))
    return round(float(stretch_index), 2)


def compute_team_length(context):
    """Feature 3: distância horizontal (eixo x) entre o defensor de linha mais atrás e o mais à frente (m)."""
    xs = context["groups"]["defending_outfield"]["xs"]
    if len(xs) == 0:
        return np.nan
    return round(float(xs.max() - xs.min()), 2)


def compute_team_width(context):
    """
    Feature 8: distância vertical (eixo y) entre o defensor de linha mais
    próximo de cada lateral do campo (extremos superior/inferior em y) —
    mesmo conjunto de jogadores de compute_team_length (defensores de linha,
    sem GK), só que no eixo y em vez de x.
    """
    ys = context["groups"]["defending_outfield"]["ys"]
    if len(ys) == 0:
        return np.nan
    return round(float(ys.max() - ys.min()), 2)


def compute_defense_width(context):
    """
    Feature 9: igual a compute_team_width, mas só considera jogadores com
    position.groupType == "D" (zagueiros/laterais) — não todo o time
    defendendo, só a linha de fundo "de verdade".
    """
    ys = context["groups"]["defense_line"]["ys"]
    if len(ys) == 0:
        return np.nan
    return round(float(ys.max() - ys.min()), 2)


# ---------------------------------------------------------------------------
# Altura da defesa em relação ao próprio gol
# ---------------------------------------------------------------------------

def compute_height_goal_player(context):
    """
    Feature 4: distância horizontal (eixo x) entre o gol do time defendendo e
    o defensor de linha mais próximo desse gol — mede a altura da linha
    defensiva. Ataque sempre normalizado pra direita (target_engineering.ipynb),
    logo o gol do time defendendo fica sempre em x = +stadiumLength/2, e o
    defensor mais próximo dele é o de maior x.
    """
    xs = context["groups"]["defending_outfield"]["xs"]
    if len(xs) == 0 or pd.isna(context["stadiumLength"]):
        return np.nan
    goal_x = context["stadiumLength"] / 2
    return round(float(goal_x - xs.max()), 2)


def compute_height_goal_team_centroide(context):
    """
    Feature 5: distância horizontal (eixo x) entre o gol do time defendendo e
    o centroide dos defensores de linha (mesmo centroide usado em
    compute_stretch_index) — versão do height_goal_player usando a posição
    média do time em vez do defensor mais recuado. Mesma normalização: gol
    do time defendendo sempre em x = +stadiumLength/2.
    """
    g = context["groups"]["defending_outfield"]
    if len(g["xs"]) == 0 or pd.isna(context["stadiumLength"]):
        return np.nan
    goal_x = context["stadiumLength"] / 2
    return round(float(goal_x - g["cx"]), 2)


def compute_height_goal_def_centroide(context):
    """
    Feature 13: distância horizontal (eixo x) entre o gol do time defendendo
    e o centroide da linha defensiva (groupType == "D", sem GK) — mesma
    lógica de compute_height_goal_team_centroide, mas usando só o centroide
    dos zagueiros/laterais em vez do time inteiro.
    """
    cx_def = context["groups"]["defense_line"]["cx"]
    if np.isnan(cx_def) or pd.isna(context["stadiumLength"]):
        return np.nan
    goal_x = context["stadiumLength"] / 2
    return round(float(goal_x - cx_def), 2)


# ---------------------------------------------------------------------------
# Distância longitudinal entre linhas (centroides por groupType)
# ---------------------------------------------------------------------------

def compute_def_mid_dist(context):
    """Feature 10: distância longitudinal (eixo x) entre o centroide da linha defensiva (groupType D) e o centroide do meio-campo (groupType M)."""
    cx_def = context["groups"]["defense_line"]["cx"]
    cx_mid = context["groups"]["mid_line"]["cx"]
    if np.isnan(cx_def) or np.isnan(cx_mid):
        return np.nan
    return round(float(abs(cx_def - cx_mid)), 2)


def compute_def_atk_dist(context):
    """Feature 11: distância longitudinal (eixo x) entre o centroide da linha defensiva (groupType D) e o centroide do ataque (groupType A)."""
    cx_def = context["groups"]["defense_line"]["cx"]
    cx_atk = context["groups"]["atk_line"]["cx"]
    if np.isnan(cx_def) or np.isnan(cx_atk):
        return np.nan
    return round(float(abs(cx_def - cx_atk)), 2)


def compute_atk_mid_dist(context):
    """Feature 12: distância longitudinal (eixo x) entre o centroide do ataque (groupType A) e o centroide do meio-campo (groupType M)."""
    cx_atk = context["groups"]["atk_line"]["cx"]
    cx_mid = context["groups"]["mid_line"]["cx"]
    if np.isnan(cx_atk) or np.isnan(cx_mid):
        return np.nan
    return round(float(abs(cx_atk - cx_mid)), 2)


# ---------------------------------------------------------------------------
# Superioridade numérica ao redor da bola
# ---------------------------------------------------------------------------

def _count_within_radius(xs, ys, ball_x, ball_y, radius):
    return int(((xs - ball_x) ** 2 + (ys - ball_y) ** 2 <= radius ** 2).sum())


def _numeric_superiority(context, radius):
    bx, by = context["ball_x"], context["ball_y"]
    if pd.isna(bx) or pd.isna(by):
        return np.nan
    defending = context["groups"]["defending"]
    attacking = context["groups"]["attacking"]
    defenders = _count_within_radius(defending["xs"], defending["ys"], bx, by, radius)
    attackers = _count_within_radius(attacking["xs"], attacking["ys"], bx, by, radius)
    return defenders - attackers


def compute_numeric_superiority_10m(context):
    """Feature 6: defensores menos atacantes num raio de 10m da bola."""
    return _numeric_superiority(context, radius=10)


def compute_numeric_superiority_20m(context):
    """Feature 7: defensores menos atacantes num raio de 20m da bola."""
    return _numeric_superiority(context, radius=20)


# nome da coluna (bate com FEATURE_COLUMNS) -> função que calcula ela
FEATURE_FUNCS = {
    # forma/dispersão
    "surface_area": compute_surface_area,
    "stretch_index": compute_stretch_index,
    "team_length": compute_team_length,
    "team_width": compute_team_width,
    "defense_width": compute_defense_width,

    # altura em relação ao gol
    "height_goal_player": compute_height_goal_player,
    "height_goal_team_centroide": compute_height_goal_team_centroide,
    "height_goal_def_centroide": compute_height_goal_def_centroide,

    # distância entre linhas
    "def_mid_dist": compute_def_mid_dist,
    "def_atk_dist": compute_def_atk_dist,
    "atk_mid_dist": compute_atk_mid_dist,

    # superioridade numérica
    "numeric_superiority_10m": compute_numeric_superiority_10m,
    "numeric_superiority_20m": compute_numeric_superiority_20m,
}

In [5]:
def compute_defensive_features(iterator):
    """
    Função a ser passada pro mapInPandas: roda por lote (batch) de cada
    partição via Arrow nos executors — nunca materializa o df inteiro no
    driver, ao contrário de toPandas(). Só orquestra: monta o context de
    cada linha (dinamicamente a partir de TEMP_COLS, com os grupos de
    GROUP_DEFS já limpos/com centroide calculados uma única vez) e chama
    cada FEATURE_FUNCS — o cálculo em si vive nas funções compute_*, então
    isso não precisa mudar conforme mais features/colunas temporárias são
    adicionadas. Depende de TEMP_COLS, FEATURE_COLUMNS e GROUP_DEFS
    (definidas mais adiante, depois que o df é montado) já existirem no
    escopo global quando for chamada.

    Parâmetros
    ----------
    iterator : Iterator[pandas.DataFrame]
        Lotes de linhas de uma partição, cada um já com as colunas
        temporárias achatadas (TEMP_COLS).

    Yields
    ------
    pandas.DataFrame
        Cada lote de entrada, sem as colunas temporárias, com uma coluna a
        mais por feature em FEATURE_COLUMNS.
    """
    for pdf in iterator:
        n = len(pdf)
        results = {name: np.full(n, np.nan) for name, _ in FEATURE_COLUMNS}

        # uma lista por coluna temporária, pra indexar por linha sem
        # depender de saber de antemão quais colunas cada feature usa
        temp_values = {col: pdf[col].tolist() for col in TEMP_COLS}

        for i in range(n):
            raw = {col: values[i] for col, values in temp_values.items()}
            groups = {
                group_name: _clean_group(raw[x_key], raw[y_key])
                for group_name, x_key, y_key in GROUP_DEFS
            }
            context = {"groups": groups, "ball_x": raw["ball_x"], "ball_y": raw["ball_y"], "stadiumLength": raw["stadiumLength"]}
            for name, func in FEATURE_FUNCS.items():
                results[name][i] = func(context)

        pdf = pdf.drop(columns=TEMP_COLS)
        for name, values in results.items():
            pdf[name] = values

        yield pdf

In [6]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

In [7]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")

df = spark.read.parquet(threat_dataset_path)

In [8]:
# Sanity check: concentração dos defensores (eixo x) por groupType/type de
# posição — pega só defendingPlayersNorm (é o que as features realmente
# usam) e tira média/mediana/desvio de x por grupo/posição, pra confirmar
# que o mapeamento de posições faz sentido tático (D mais recuado, A mais
# adiantado, etc.)
df_players_x = (
    df.select(F.explode("defendingPlayersNorm").alias("p"))
    .select(
        F.col("p.position.groupType").alias("groupType"),
        F.col("p.position.type").alias("type"),
        F.col("p.position.typeDescription").alias("typeDescription"),
        F.col("p.x").alias("x"),
    )
)

(
    df_players_x
    .groupBy("groupType", "type", "typeDescription")
    .agg(
        F.round(F.mean("x"), 2).alias("avg_x"),
        F.round(F.median("x"), 2).alias("median_x"),
        F.round(F.stddev("x"), 2).alias("std_x"),
        F.count("*").alias("qtd"),
    )
    .orderBy("groupType", "avg_x")
    .show(50, truncate=False)
)

+---------+----+--------------------+-----+--------+-----+------+
|groupType|type|typeDescription     |avg_x|median_x|std_x|qtd   |
+---------+----+--------------------+-----+--------+-----+------+
|A        |CF  |Centre-Forward      |-4.08|-3.86   |21.35|512883|
|A        |LW  |Left Winger         |0.64 |0.5     |21.9 |345901|
|A        |RW  |Right Winger        |1.84 |1.84    |22.04|301610|
|D        |LWB |Left Wing-Back      |13.45|14.64   |19.81|58340 |
|D        |LB  |Left Back           |14.59|16.07   |19.84|314385|
|D        |RB  |Right Back          |14.68|15.81   |19.76|359719|
|D        |RWB |Right Wing-Back     |16.24|17.27   |17.41|13231 |
|D        |MCB |Middle Centre-Back  |16.89|16.76   |17.76|65080 |
|D        |LCB |Left Centre-Back    |17.55|17.97   |18.01|542164|
|D        |RCB |Right Centre-Back   |17.72|18.2    |17.76|445375|
|D        |GK  |Goalkeeper          |39.95|40.49   |7.91 |448786|
|M        |AM  |Attacking Midfielder|1.41 |1.25    |21.6 |312841|
|M        

- Essa análise serviu para averiguar se as posições dos jogadores nas partidas possuem uma média/mediana do eixo x que faça sentido conforme a sua posição (ex: Laterais e zagueiros terem um eixo x consistente na região da zaga). 
- Olhando em uma visão geral da temporada todos os eventos e agregações do eixo x entre as posições do time no momento que estão defendendo parecem estar fazendo sentido e confiáveis de serem usados. 
- Considerando que o gol defendido está sempre à direita, é possivel ver, por exemplo, uma simetria entre jogadores da lateral/zagueiro na esquerda/direita com uma média bem parecida.

In [9]:
# Só o necessário pro cálculo das features
df = df.select('competitionId', 'season', 'gameId', 'eventId', 'attackingPlayersNorm', 'defendingPlayersNorm', 'ballsNorm', 'stadiumLength')

df.show(2)

+-------------+---------+------+--------------------+--------------------+--------------------+--------------------+-------------+
|competitionId|   season|gameId|             eventId|attackingPlayersNorm|defendingPlayersNorm|           ballsNorm|stadiumLength|
+-------------+---------+------+--------------------+--------------------+--------------------+--------------------+-------------+
|            1|2022-2023|  4436|0ac865b9dc18eeffe...|[{-22.613, 14.737...|[{-43.32, -0.751,...|[{-46.84, -7.1, 1...|        101.0|
|            1|2022-2023|  4436|ad4695b2b59ac8f4c...|[{22.188, 21.917,...|[{0.965, -5.927, ...|[{22.2, -9.2, 0.0...|        101.0|
+-------------+---------+------+--------------------+--------------------+--------------------+--------------------+-------------+
only showing top 2 rows


In [10]:
df.count()

448193

In [11]:
# Achata array<struct> em array<float> de x/y antes do mapInPandas — reduz o
# payload serializado via Arrow (dropa player.id/name/visibility/confidence/
# position, que não entram nos cálculos). GK já sai filtrado aqui pro
# defendingOutfield, usado nas Features 1/2 (casco/stretch); defending_x/y
# e attacking_x/y (com GK) e ball_x/y alimentam as Features 6/7.
# position agora é um struct (type/typeDescription/groupType) — usa .type
# pra comparar com "GK", igual ao "GK" que já vinha da sigla original.
defending_outfield = F.filter("defendingPlayersNorm", lambda p: p["position"]["type"] != "GK")


def _max_shared_coord_count(players_col):
    """
    Maior quantidade de jogadores compartilhando exatamente a mesma
    coordenada (x, y) — mesmo critério validado em
    sanity_check_features_tracking.ipynb pra detectar tracking inconsistente
    (provider "perde" jogadores e estima todos na mesma posição). Versão
    Spark-nativa (roda no achatamento, antes do mapInPandas).
    """
    coords = F.transform(players_col, lambda p: F.struct(p["x"].alias("x"), p["y"].alias("y")))
    distinct_coords = F.array_distinct(coords)
    group_sizes = F.transform(distinct_coords, lambda c: F.size(F.filter(coords, lambda y: y.eqNullSafe(c))))
    return F.array_max(group_sizes)


# Flag de qualidade do tracking: >= 2 defensores de linha na EXATA mesma
# coordenada já é fisicamente implausível (validado em
# sanity_check_features_tracking.ipynb). Fica marcado, não filtrado — quem
# consumir features_dataset decide se remove ou não.
is_tracking_inconsistent = _max_shared_coord_count(defending_outfield) >= 2

# Linha de fundo "de verdade": groupType == "D" (zagueiros/laterais) e não
# GK — GK também tem groupType "D" em POSITION_INFO, então precisa excluir
# explicitamente. Usada em defense_width, que (diferente de team_width) só
# considera os jogadores da posição defensiva, não todo o time defendendo.
defense_line = F.filter(
    "defendingPlayersNorm",
    lambda p: (p["position"]["groupType"] == "D") & (p["position"]["type"] != "GK")
)

# Linhas de meio e ataque do time defendendo, por groupType — usadas pros
# centroides de def_mid_long_dist/def_atk_long_dist/mid_atk_long_dist
mid_line = F.filter("defendingPlayersNorm", lambda p: p["position"]["groupType"] == "M")
atk_line = F.filter("defendingPlayersNorm", lambda p: p["position"]["groupType"] == "A")

df = df.withColumns({
    "is_tracking_inconsistent": is_tracking_inconsistent,
    "defending_outfield_x": F.transform(defending_outfield, lambda p: p["x"]),
    "defending_outfield_y": F.transform(defending_outfield, lambda p: p["y"]),
    "defense_line_x": F.transform(defense_line, lambda p: p["x"]),
    "defense_line_y": F.transform(defense_line, lambda p: p["y"]),
    "mid_line_x": F.transform(mid_line, lambda p: p["x"]),
    "mid_line_y": F.transform(mid_line, lambda p: p["y"]),
    "atk_line_x": F.transform(atk_line, lambda p: p["x"]),
    "atk_line_y": F.transform(atk_line, lambda p: p["y"]),
    "defending_x": F.transform("defendingPlayersNorm", lambda p: p["x"]),
    "defending_y": F.transform("defendingPlayersNorm", lambda p: p["y"]),
    "attacking_x": F.transform("attackingPlayersNorm", lambda p: p["x"]),
    "attacking_y": F.transform("attackingPlayersNorm", lambda p: p["y"]),
    "ball_x": F.get("ballsNorm", 0)["x"],
    "ball_y": F.get("ballsNorm", 0)["y"],
}).drop("attackingPlayersNorm", "defendingPlayersNorm", "ballsNorm")

df.printSchema()

root
 |-- competitionId: long (nullable = true)
 |-- season: string (nullable = true)
 |-- gameId: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- stadiumLength: float (nullable = true)
 |-- is_tracking_inconsistent: boolean (nullable = true)
 |-- defending_outfield_x: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- defending_outfield_y: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- defense_line_x: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- defense_line_y: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- mid_line_x: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- mid_line_y: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- atk_line_x: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- atk_line_y: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- d

In [12]:
# Colunas temporárias (arrays achatados + stadiumLength) criadas só pra
# alimentar o cálculo das features — descartadas do schema de saída do
# mapInPandas, que fica só com eventId + as features
TEMP_COLS = [
    "defending_outfield_x", "defending_outfield_y",
    "defense_line_x", "defense_line_y",
    "mid_line_x", "mid_line_y",
    "atk_line_x", "atk_line_y",
    "defending_x", "defending_y",
    "attacking_x", "attacking_y",
    "ball_x", "ball_y",
    "stadiumLength",
]

# Uma linha por feature: nome da coluna de saída + tipo Spark. Pra adicionar
# uma feature nova: escreve a função dela na célula de baixo, adiciona uma
# linha aqui e uma entrada em FEATURE_FUNCS — o resto (schema, orquestração)
# não muda. Agrupado por tema pra facilitar leitura.
FEATURE_COLUMNS = [
    # Forma/dispersão do time defendendo
    ("surface_area", DoubleType()),    # Feature 1 — convex hull
    ("stretch_index", DoubleType()),   # Feature 2 — stretch index
    ("team_length", DoubleType()),     # Feature 3 — comprimento (x) do time
    ("team_width", DoubleType()),      # Feature 4 — largura (y) do time
    ("defense_width", DoubleType()),   # Feature 5 — largura (y) da linha defensiva (groupType == "D")

    # Altura da defesa em relação ao próprio gol
    ("height_goal_player", DoubleType()),          # Feature 6 — gol -> defensor mais recuado
    ("height_goal_team_centroide", DoubleType()),  # Feature 7 — gol -> centroide do time
    ("height_goal_def_centroide", DoubleType()),   # Feature 8 — gol -> centroide da linha defensiva

    # Distância longitudinal entre linhas (centroides por groupType)
    ("def_mid_dist", DoubleType()),  # Feature 9 — centroide defesa <-> meio
    ("def_atk_dist", DoubleType()),  # Feature 10 — centroide defesa <-> ataque
    ("atk_mid_dist", DoubleType()),  # Feature 11 — centroide ataque <-> meio

    # Superioridade numérica ao redor da bola
    ("numeric_superiority_10m", DoubleType()),  # Feature 12
    ("numeric_superiority_20m", DoubleType()),  # Feature 13
]

# Schema de saída = eventId (única coluna que sobra de df.schema depois de
# tirar TEMP_COLS) + as features
FEATURE_SCHEMA = StructType(
    [f for f in df.schema.fields if f.name not in TEMP_COLS]
    + [StructField(name, dtype) for name, dtype in FEATURE_COLUMNS]
)

In [13]:
df.rdd.getNumPartitions()

15

In [ ]:
df = df.repartition(150)
df_features = df.mapInPandas(compute_defensive_features, schema=FEATURE_SCHEMA)

In [15]:
# output já é só eventId + as features (schema enxuto)
df_features.show(10, truncate=False)

+-------------+---------+------+--------------------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+
|competitionId|season   |gameId|eventId                         |is_tracking_inconsistent|surface_area|stretch_index|team_length|team_width|defense_width|height_goal_player|height_goal_team_centroide|height_goal_def_centroide|def_mid_dist|def_atk_dist|atk_mid_dist|numeric_superiority_10m|numeric_superiority_20m|
+-------------+---------+------+--------------------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+
|1            |2022-2023|4669  |757c63f63f2751492e9fe93e46

In [16]:
output_path = str(Path().resolve().parent.parent / "data" / "features_dataset")
df_features.write.mode("overwrite").option("header", True).csv(output_path)